<a href="https://colab.research.google.com/github/aldenbarnes/structure_properties_fall_2026/blob/main/Code/address_comps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Address → Rental Comps

Run the cells from top to bottom.

1. **Type an address** in Step 1.
2. **Step 2** looks the address up in San Francisco's official address database (DataSF) and finds its
   coordinates, neighborhood, supervisor district and **zip code**.
3. **Step 3** asks RealtyAPI for rental listings. It makes **exactly two requests**:
   * up to 500 listings from **all of San Francisco**
   * up to 500 listings from the address's **zip code**
4. **Step 4** saves the results as CSV files.

RealtyAPI charges credits per request, so the notebook stops itself after two requests. Re-running Step 3
will **not** send new requests; restart the runtime if you really want to pull again.

## Step 0: Setup
Add your RealtyAPI key to Colab once: click the **🔑 key icon** in the left sidebar, add a secret named
`REALTYAPI_KEY`, and turn on notebook access. If you skip this, Step 3 will ask you to paste the key.

In [ ]:
import json
import re
from datetime import datetime

import pandas as pd
import requests

pd.set_option("display.max_columns", 50)

## Step 1: Enter the address
Street number and street name are enough. Unit numbers, city, state and zip are fine too; they get cleaned off.

In [ ]:
ADDRESS = "1010 Anza St"   # <-- change me

## Step 2: Look up the address in the San Francisco database
This downloads DataSF's **Addresses with Units (Enterprise Addressing System)** table, the same file used in
`test_colab.ipynb`. It is a big file, so this cell takes a minute.

In [ ]:
SF_ADDRESSES_URL = "https://data.sf.gov/api/v3/views/ramy-di5m/export.csv?accessType=DOWNLOAD"

sf_addresses = pd.read_csv(SF_ADDRESSES_URL, dtype=str)
print(f"Loaded {len(sf_addresses):,} San Francisco addresses")
print("Columns:", list(sf_addresses.columns))

### Clean up the address so it matches the database format
The database writes addresses like `1010 ANZA ST`, so we upper-case the input, drop the city/state/zip,
drop any unit number, and shorten words like `STREET` → `ST`.

In [ ]:
STREET_TYPES = {
    "STREET": "ST", "AVENUE": "AVE", "BOULEVARD": "BLVD", "DRIVE": "DR", "ROAD": "RD",
    "PLACE": "PL", "COURT": "CT", "LANE": "LN", "TERRACE": "TER", "WAY": "WAY",
    "ALLEY": "ALY", "CIRCLE": "CIR", "HIGHWAY": "HWY", "PLAZA": "PLZ", "PARKWAY": "PKWY",
}


def clean_address(text):
    """'1010 Anza Street, Apt 2, San Francisco, CA 94118' -> '1010 ANZA ST'"""
    text = text.upper().split(",")[0]                  # keep only the street part
    text = re.sub(r"\s+(APT|UNIT|STE|SUITE|#).*$", "", text)  # drop unit numbers
    text = re.sub(r"[^A-Z0-9 ]", " ", text)             # drop punctuation
    words = [STREET_TYPES.get(w, w) for w in text.split()]
    return " ".join(words)


search_address = clean_address(ADDRESS)
print("Searching for:", search_address)

### Find the matching row
First we look for an exact match. If there isn't one, we look for addresses that start with what you typed
and show the options.

In [ ]:
def find_column(df, *keywords):
    """Return the first column whose name contains any keyword (case-insensitive)."""
    for keyword in keywords:
        for col in df.columns:
            if keyword.lower() in col.lower():
                return col
    return None


ADDRESS_COL = "Address"
ZIP_COL = find_column(sf_addresses, "zip")
NHOOD_COL = find_column(sf_addresses, "nhood", "neighborhood")
DISTRICT_COL = find_column(sf_addresses, "supervisor")
LAT_COL = find_column(sf_addresses, "latitude")
LON_COL = find_column(sf_addresses, "longitude")
PARCEL_COL = find_column(sf_addresses, "parcel", "blklot")

db_address = sf_addresses[ADDRESS_COL].fillna("").str.upper().str.split().str.join(" ")

matches = sf_addresses[db_address == search_address]
if matches.empty:
    matches = sf_addresses[db_address.str.startswith(search_address)]

if matches.empty:
    raise ValueError(f"No San Francisco address found for '{search_address}'. Check the spelling in Step 1.")

print(f"{len(matches)} matching row(s) (one per unit is normal)")
matches.head(10)

### The address's geography
If more than one building matched, change `ADDRESS` in Step 1 to the exact one you want and re-run.

In [ ]:
unique_addresses = matches[ADDRESS_COL].unique()
if len(unique_addresses) > 1:
    print("More than one address matched. Using the first one:")
    for a in unique_addresses[:10]:
        print("   ", a)

match = matches[matches[ADDRESS_COL] == unique_addresses[0]]
first = match.iloc[0]

def value(col):
    return first[col] if col else None

# Zip codes: use the most common one for this address, falling back to a zip typed in Step 1
zip_code = match[ZIP_COL].dropna().mode().iloc[0] if ZIP_COL and match[ZIP_COL].notna().any() else None
if zip_code is None:
    typed_zip = re.search(r"\b(94\d{3})\b", ADDRESS)
    zip_code = typed_zip.group(1) if typed_zip else None
zip_code = str(zip_code).split(".")[0][:5] if zip_code else None

property_info = {
    "address": first[ADDRESS_COL],
    "zip_code": zip_code,
    "neighborhood": value(NHOOD_COL),
    "supervisor_district": value(DISTRICT_COL),
    "latitude": value(LAT_COL),
    "longitude": value(LON_COL),
    "parcel": value(PARCEL_COL),
    "units_on_record": len(match),
}

if not zip_code:
    raise ValueError("Could not find a zip code. Add it to ADDRESS in Step 1, e.g. '1010 Anza St, 94118'.")

pd.Series(property_info, name="Your property").to_frame()

## Step 3: Pull rental listings from RealtyAPI (2 requests total)
Uses the same `/search/bylocation` endpoint as `realtyapi_notebook.ipynb`. 500 is the most one request can
return.

In [ ]:
REALTYAPI_URL = "https://apartments.realtyapi.io/search/bylocation"
RESULTS_PER_REQUEST = 500
MAX_REQUESTS = 2

# Load the key from Colab secrets, or ask for it
try:
    from google.colab import userdata
    REALTYAPI_KEY = userdata.get("REALTYAPI_KEY")
except Exception:
    import getpass
    REALTYAPI_KEY = getpass.getpass("RealtyAPI key (hidden): ").strip()

# Keeps track of requests so re-running cells can't spend extra credits
if "api_responses" not in globals():
    api_responses = {}


def search_listings(label, location):
    """Ask RealtyAPI for up to 500 listings in `location`. Each label is only ever requested once."""
    if label in api_responses:
        print(f"[{label}] already downloaded, reusing it (no new request).")
        return api_responses[label]
    if len(api_responses) >= MAX_REQUESTS:
        raise RuntimeError(f"Stopped: already made {MAX_REQUESTS} requests this session.")

    print(f"[{label}] requesting up to {RESULTS_PER_REQUEST} listings for '{location}'...")
    response = requests.get(
        REALTYAPI_URL,
        params={"location": location, "page": 1, "resultCount": RESULTS_PER_REQUEST},
        headers={"x-realtyapi-key": REALTYAPI_KEY, "Accept": "application/json"},
        timeout=120,
    )
    if not response.ok:
        # Failed requests still count toward the cap, in case they cost credits
        api_responses[label] = {"error": f"HTTP {response.status_code}"}
        raise RuntimeError(f"[{label}] RealtyAPI returned HTTP {response.status_code}. Check your key and credits.")
    api_responses[label] = response.json()

    data = api_responses[label]
    print(f"[{label}] got {len(data.get('searchResults', []))} listings "
          f"(RealtyAPI reports {data.get('total')} total matches)")
    return data

### Request 1 of 2: all of San Francisco

In [ ]:
sf_response = search_listings("san_francisco", "San Francisco, CA")

### Request 2 of 2: the property's zip code

In [ ]:
zip_response = search_listings("zip_code", f"San Francisco, CA {zip_code}")

### Turn the responses into tables
Each row is one listing (a building or unit for rent). Nested fields are flattened into columns like
`address.postalCode`.

In [ ]:
def to_table(response, search_name):
    df = pd.json_normalize(response.get("searchResults", []))
    df.insert(0, "search", search_name)
    return df


sf_listings = to_table(sf_response, "San Francisco")
zip_listings = to_table(zip_response, f"Zip {zip_code}")

print(f"San Francisco listings: {len(sf_listings)}")
print(f"Zip {zip_code} listings:  {len(zip_listings)}")

PREVIEW_COLUMNS = ["search", "name", "oneLineAddress", "address.postalCode",
                   "propertyType", "bedRange", "rentRange", "latitude", "longitude", "listingKey"]
zip_listings[[c for c in PREVIEW_COLUMNS if c in zip_listings]].head(20)

In [ ]:
sf_listings[[c for c in PREVIEW_COLUMNS if c in sf_listings]].head(20)

## Step 4: Save the results
Files go in `realtyapi_output/` with the address and date in the name. In Colab, open the 📁 folder icon on
the left to download them. The raw JSON is saved too, so nothing is lost and nothing needs re-requesting.

In [ ]:
from pathlib import Path

output_dir = Path("realtyapi_output")
output_dir.mkdir(exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M")
slug = search_address.replace(" ", "_")

all_listings = pd.concat([sf_listings, zip_listings], ignore_index=True)
all_listings["subject_address"] = property_info["address"]
all_listings["subject_zip"] = zip_code

all_listings.to_csv(output_dir / f"{slug}_{stamp}_listings.csv", index=False)
pd.Series(property_info).to_csv(output_dir / f"{slug}_{stamp}_property.csv", header=["value"])
(output_dir / f"{slug}_{stamp}_raw.json").write_text(json.dumps(api_responses, indent=2))

print("Saved:")
for f in sorted(output_dir.glob(f"{slug}_{stamp}_*")):
    print("   ", f)
print(f"Requests used this session: {len(api_responses)} of {MAX_REQUESTS}")